# 3장 3강: Gradient Descent와 Learning Rate — 실습문제

## 실습 목표

- 주어진 기울기 공식에 숫자를 넣어 가중치와 편향을 바꿉니다.
- 학습률에 따라 이동하는 양과 손실이 어떻게 달라지는지 읽습니다.
- 전체 데이터와 작은 묶음을 사용할 때의 업데이트를 구별합니다.
- 반복 계산에서 수렴의 뜻을 이해하고, 검증 손실 표에서 학습 종료 시점을 찾습니다.

## 실습 환경 / 데이터

Python과 NumPy를 사용합니다. 외부 데이터 파일은 필요하지 않습니다. 제공된 「3장 3강: Gradient Descent와 Learning Rate」의 본문과 연습문제에 나온 식·숫자만 사용합니다.

## 실습 구성과 교안 대응

| 구분 | 실습 내용 | 교안 근거 |
|---|---|---|
| 필수 1 / 문제 1-1 | w와 b를 두 번 업데이트하기 | 1절 본문: x=2, y=10, w=3, b=1, 학습률 0.05 |
| 필수 1 / 문제 1-2 | 네 학습률의 3회 업데이트 결과 읽기 | 2절 본문: L=(10−2w)², 초기 w=3의 학습률 비교 |
| 필수 2 / 문제 2-1 | Batch와 Mini-Batch의 기본 계산 | 3절 본문 및 연습문제 3: X=[1,2,3,4], y=[2,4,6,8] |
| 필수 2 / 문제 2-2 | 제공 코드로 수렴 관찰하기 | 3절 본문: L=(w−5)², 초기 w=1, 학습률 0.1, 30회 업데이트 |
| 과제 / 문제 3-1 | 같은 계산 순서로 두 번 업데이트하기 | 연습문제 1: x=3, y=12, w=2, b=1, 학습률 0.02 |
| 과제 / 문제 3-2 | 검증 손실 표에서 최적 시점과 종료 시점 찾기 | 연습문제 3의 문제 6: 6개 Epoch의 손실 표, patience=3 |

**교안의 연습문제 전체를 그대로 옮긴 자료는 아닙니다.** 본문 예제와 연습문제 일부를 기초 수준의 필수 실습 4문제와 복습 과제 2문제로 재구성했습니다. 사용한 개념과 계산 방법은 해당 강의의 범위 안에 있습니다.

## 진행 안내

1. 실습 준비 셀을 먼저 실행합니다.
2. 문제에 제공된 공식을 보고 숫자를 대입합니다. 미분 공식을 혼자 유도할 필요는 없습니다.
3. 문제 1-2와 2-2의 코드는 제공되어 있습니다. 실행한 뒤 표와 출력을 읽어 답하면 됩니다.
4. 다른 문제도 손으로 계산한 값을 코드로 확인하면 됩니다. 긴 프로그램을 직접 설계할 필요는 없습니다.
5. 각 문제의 Q1~Q4에 답합니다. 과제는 앞에서 배운 계산과 표 읽기만 복습합니다.
6. 소수는 계산 도중 반올림하지 않습니다. 출력에 긴 소수가 보이면 마지막에만 반올림합니다.

## 먼저 읽는 용어와 기호

| 용어·기호 | 쉬운 뜻 |
|---|---|
| 모델 / 예측 | 입력으로 정답을 추측하는 계산 규칙 / 그 규칙으로 계산한 값 |
| x / y | 입력 / 실제 정답. 여기서는 문제에서 주어지는 고정값 |
| X | 여러 입력을 순서대로 묶은 것 |
| ŷ / y_pred | 모델이 계산한 예측값. ŷ는 ‘와이 햇’으로 읽음 |
| w / b | 입력에 곱하는 가중치 / 예측에 더하는 편향 |
| 파라미터 | 학습하면서 조절하는 숫자. 이 실습에서는 w 또는 w와 b |
| L / loss | 예측과 정답의 차이를 평가하는 손실. 이 자료의 손실은 작을수록 오차가 작음 |
| MSE | 각 데이터의 ‘정답−예측’을 제곱한 뒤 평균 낸 손실 |
| 기울기 / grad | 현재 파라미터를 조금 바꿀 때 손실이 변하는 방향과 정도 |
| Gradient / ∇L | 여러 파라미터의 손실 기울기를 순서대로 묶은 것 |
| α / learning_rate | 학습률. ‘알파’라고 읽으며, 기울기에 곱해 업데이트 크기를 조절하는 양수 |
| 업데이트 | 현재 파라미터에서 ‘학습률×기울기’를 빼 새 값으로 바꾸는 일 |
| old / new | 업데이트 전 / 업데이트 후 |
| Batch / Mini-Batch / SGD | 전체 데이터 / 작은 데이터 묶음 / 데이터 하나로 기울기를 구해 업데이트하는 방식 |
| Epoch | 전체 학습 데이터를 한 번 사용한 단위. ‘에포크’라고 읽음 |
| Iteration | 파라미터를 한 번 업데이트하는 단위. ‘이터레이션’이라고 읽음 |
| 수렴 / Convergence | 반복하면서 파라미터나 손실이 일정한 값에 가까워지는 현상 |
| 학습 손실 / 검증 손실 | 학습에 사용하는 데이터의 손실 / 별도로 둔 검증 데이터의 손실 |
| Early Stopping / patience | 검증 손실이 좋아지지 않으면 학습을 일찍 멈추는 방법 / 연속으로 기다릴 횟수 |

### 수식을 읽는 방법

- `ŷ=wx+b`는 **w와 x를 곱한 뒤 b를 더한다**는 뜻입니다. 붙여 쓴 wx는 곱셈입니다.
- `L=(y−ŷ)²`는 **정답에서 예측을 뺀 값을 자기 자신과 곱한다**는 뜻입니다. 예를 들어 (10−7)²=3×3=9입니다.
- `∂L/∂w`는 w에 대한 손실의 기울기입니다. 기호가 분수처럼 생겼다고 해서 L을 w로 나누라는 뜻은 아닙니다. 코드에서는 `grad_w`라고 씁니다.
- `w_new=w_old−α×grad_w`는 **현재 w에서 학습률과 기울기의 곱을 뺀다**는 뜻입니다.
- `3−0.05×(−12)=3−(−0.6)=3+0.6=3.6`입니다. 음수를 빼면 더하기가 됩니다.
- `≈`는 ‘대략 같다’는 뜻입니다. 표의 긴 소수는 보기 좋게 반올림한 값일 수 있습니다.

학습률이 0.05라고 해서 언제나 0.05만큼 이동하지는 않습니다. 실제 변화량은 **−학습률×현재 기울기**입니다. 기울기를 계산하는 과정과 파라미터를 실제로 바꾸는 과정도 구별하세요.

## 실습 준비

아래 셀을 먼저 실행하세요. NumPy는 여러 숫자를 묶어서 계산하도록 돕는 도구입니다. `np`는 NumPy를 짧게 부르는 이름입니다.

In [ ]:
import numpy as np

---

## 필수 실습 1: 업데이트와 학습률

### 문제 1-1. 가중치와 편향을 두 번 업데이트하기

### 문제 설명

모델은 ŷ=wx+b입니다. x=2, y=10, 처음 w=3, b=1, 학습률 α=0.05를 사용합니다. 업데이트는 정확히 두 번만 합니다.

### 제공 공식

| 순서 | 계산 |
|---|---|
| 1 | 예측 ŷ=wx+b |
| 2 | 손실 L=(y−ŷ)² |
| 3 | w의 기울기 g_w=2(ŷ−y)x, b의 기울기 g_b=2(ŷ−y) |
| 4 | 새 w=w−0.05×g_w, 새 b=b−0.05×g_b |
| 5 | 새 w와 b로 예측과 손실을 다시 계산 |

**두 기울기는 같은 업데이트 전 상태에서 모두 계산합니다.** 그다음 두 값을 바꿉니다. 두 번째 업데이트에서는 새 상태의 기울기를 다시 구하세요.

### 요구사항

- 초기 예측·손실·두 기울기를 계산합니다.
- 1회와 2회 업데이트 후 w, b, 예측, 손실을 계산합니다.
- 공식에 대입한 값을 적고 Q1~Q4에 답합니다.

#### 해석 질문

**Q1.** 업데이트 전 예측, 손실, 두 기울기는 얼마인가요?

**Q2.** 첫 업데이트 후 w, b, 예측, 손실은 얼마인가요?

**Q3.** 두 번째 업데이트에서 사용할 기울기와 업데이트 후 결과는 무엇인가요?

**Q4.** 기울기는 언제 구하고, 업데이트 후 손실은 언제 구하나요?

#### 제출 결과

- 요구사항의 계산 결과 또는 제공 코드의 실행 결과
- Q1~Q4 답변

In [ ]:
# 문제 1-1. 가중치와 편향을 두 번 업데이트하기
# 1. 문제의 초기값을 설정하세요.
# 2. 제공 공식에 숫자를 대입하세요.
# 3. 새 값과 결과를 출력하세요.

#### 답변 작성란

**Q1.** 업데이트 전 예측, 손실, 두 기울기는 얼마인가요?

답변: 

**Q2.** 첫 업데이트 후 w, b, 예측, 손실은 얼마인가요?

답변: 

**Q3.** 두 번째 업데이트에서 사용할 기울기와 업데이트 후 결과는 무엇인가요?

답변: 

**Q4.** 기울기는 언제 구하고, 업데이트 후 손실은 언제 구하나요?

답변:

### 문제 1-2. 학습률에 따른 세 번의 이동 비교하기

### 문제 설명

이번에는 편향 b 없이 ŷ=2w를 사용합니다. 손실은 L=(10−2w)²이고 기울기는 g=8w−40입니다. 각각 처음 w=3에서 시작하여 학습률 0.05, 0.2, 0.25, 0.3으로 세 번씩 업데이트합니다.

### 제공 공식

- 새 w = 현재 w − 학습률×(8×현재 w−40)
- 새 손실 = (10−2×새 w)²
- w=5이면 예측이 10이 되어 손실이 0입니다.

### 요구사항

- 아래 제공 코드를 그대로 실행합니다. 반복 코드를 새로 만들 필요는 없습니다.
- 출력의 0회는 초기 상태, 1~3회는 해당 횟수만큼 바꾼 뒤의 상태입니다.
- 학습률 0.05의 첫 업데이트만 손으로 확인하고, 나머지는 출력 표를 읽어 답합니다.
- 수렴 조건식을 유도하거나 외우지 않아도 됩니다.

#### 해석 질문

**Q1.** 처음 기울기와 학습률 0.05의 첫 업데이트 후 w, 손실은 얼마인가요?

**Q2.** 학습률 0.05와 0.2에서 w의 움직임과 손실은 어떻게 다른가요?

**Q3.** 학습률 0.25에서는 가중치와 손실이 어떻게 변하나요?

**Q4.** 학습률 0.3의 결과에서 무엇을 알 수 있나요?

#### 제출 결과

- 요구사항의 계산 결과 또는 제공 코드의 실행 결과
- Q1~Q4 답변

In [ ]:
learning_rates = [0.05, 0.2, 0.25, 0.3]
for learning_rate in learning_rates:
    w = 3.0  # 학습률마다 같은 초기값으로 다시 시작
    print("학습률:", learning_rate)
    print("0회", w, (10-2*w)**2)
    for step in range(1, 4):
        grad = 8*w-40
        w = w-learning_rate*grad
        loss = (10-2*w)**2
        print(step, "회 / w, 손실:", np.round([w, loss], 6))

#### 답변 작성란

**Q1.** 처음 기울기와 학습률 0.05의 첫 업데이트 후 w, 손실은 얼마인가요?

답변: 

**Q2.** 학습률 0.05와 0.2에서 w의 움직임과 손실은 어떻게 다른가요?

답변: 

**Q3.** 학습률 0.25에서는 가중치와 손실이 어떻게 변하나요?

답변: 

**Q4.** 학습률 0.3의 결과에서 무엇을 알 수 있나요?

답변:

---

## 필수 실습 2: 데이터 묶음과 수렴

### 문제 2-1. 전체 데이터와 작은 묶음으로 업데이트하기

### 문제 설명

입력 X=[1,2,3,4], 정답 y=[2,4,6,8], 모델 ŷ=wX를 사용합니다. 처음 w=0, 학습률 0.01입니다. 편향 b는 없습니다.

- Batch: 네 데이터를 모두 사용해 한 번 업데이트합니다.
- Mini-Batch: [1,2] 입력의 묶음, [3,4] 입력의 묶음 순서로 사용해 두 번 업데이트합니다. 각 입력의 정답은 각각 [2,4], [6,8]입니다.
- 두 방식을 비교할 때 각각 w=0에서 다시 시작합니다. 데이터 순서는 섞지 않습니다.

### 제공 공식

데이터 하나의 기울기 = 2×(w×입력−정답)×입력

묶음의 기울기 = 그 묶음에 속한 각 기울기의 평균

새 w = 현재 w−0.01×묶음의 기울기

MSE = 각 데이터의 (정답−예측)²을 구한 뒤 평균

### 요구사항

1. 초기 w=0에서 네 데이터의 기울기를 구합니다.
2. Batch의 평균 기울기, 새 w, 업데이트 후 전체 MSE를 구합니다.
3. Mini-Batch의 첫 묶음으로 업데이트한 뒤, **바뀐 w**로 두 번째 묶음의 기울기를 구해 다시 업데이트합니다. 마지막 전체 MSE는 필수 계산이 아닙니다.
4. 한 Epoch의 업데이트 횟수를 비교합니다. SGD는 데이터 하나마다 업데이트한다는 뜻만 사용하며 전체 SGD 계산 코드는 작성하지 않아도 됩니다.

배열 계산을 사용하면 `grad_each = 2*(w*X-y)*X`, 평균은 `np.mean(grad_each)`입니다. 평균은 합계를 데이터 개수로 나눈 값입니다.

#### 해석 질문

**Q1.** 초기 w=0에서 네 데이터의 기울기와 전체 평균은 얼마인가요?

**Q2.** Batch로 한 번 업데이트한 w와 전체 MSE는 얼마인가요?

**Q3.** Mini-Batch의 첫 묶음과 두 번째 묶음에서 평균 기울기와 새 w는 얼마인가요?

**Q4.** 한 Epoch 동안 Batch, SGD, 크기 2의 Mini-Batch는 각각 몇 번 업데이트하나요?

#### 제출 결과

- 요구사항의 계산 결과 또는 제공 코드의 실행 결과
- Q1~Q4 답변

In [ ]:
# 문제 2-1. 전체 데이터와 작은 묶음으로 업데이트하기
# 1. 문제의 초기값을 설정하세요.
# 2. 제공 공식에 숫자를 대입하세요.
# 3. 새 값과 결과를 출력하세요.

#### 답변 작성란

**Q1.** 초기 w=0에서 네 데이터의 기울기와 전체 평균은 얼마인가요?

답변: 

**Q2.** Batch로 한 번 업데이트한 w와 전체 MSE는 얼마인가요?

답변: 

**Q3.** Mini-Batch의 첫 묶음과 두 번째 묶음에서 평균 기울기와 새 w는 얼마인가요?

답변: 

**Q4.** 한 Epoch 동안 Batch, SGD, 크기 2의 Mini-Batch는 각각 몇 번 업데이트하나요?

답변:

### 문제 2-2. 제공 코드로 수렴 관찰하기

### 문제 설명

교안의 간단한 손실 L=(w−5)²를 사용합니다. 기울기는 g=2(w−5)이고, 처음 w=1, 학습률 0.1입니다.

**문제 1-2와 손실 식이 다릅니다.** 이번 기울기는 8w−40이 아니라 2(w−5)입니다. 문제에 제시된 식을 그대로 사용하세요.

### 요구사항

- 첫 업데이트와 두 번째 업데이트 후 w와 손실을 손으로 계산합니다.
- 아래 제공 코드를 실행해 30회 업데이트 후 값을 확인합니다.
- 30회를 손으로 계산하거나 코드를 새로 작성할 필요는 없습니다.
- w와 손실이 어떤 값에 가까워지는지 Q1~Q4에 답합니다.

새 w = 현재 w−0.1×2(현재 w−5)

새 손실 = (새 w−5)²

#### 해석 질문

**Q1.** 처음 손실과 기울기, 첫 업데이트 후 w와 손실은 얼마인가요?

**Q2.** 두 번째 업데이트의 기울기와 업데이트 후 w, 손실은 얼마인가요?

**Q3.** 30회 뒤의 w와 손실은 각각 어떤 값에 가까워지나요?

**Q4.** 이 결과에서 수렴은 무슨 뜻이며, 손실이 정확히 0이 되었나요?

#### 제출 결과

- 요구사항의 계산 결과 또는 제공 코드의 실행 결과
- Q1~Q4 답변

In [ ]:
w = 1.0
learning_rate = 0.1
print("0회 / w, 손실:", w, (w-5)**2)
for step in range(1, 31):
    grad = 2*(w-5)
    w = w-learning_rate*grad
    loss = (w-5)**2
    if step in [1, 2, 5, 10, 20, 30]:
        print(step, "회 / w, 손실:", format(w, ".6f"), format(loss, ".6f"))

#### 답변 작성란

**Q1.** 처음 손실과 기울기, 첫 업데이트 후 w와 손실은 얼마인가요?

답변: 

**Q2.** 두 번째 업데이트의 기울기와 업데이트 후 w, 손실은 얼마인가요?

답변: 

**Q3.** 30회 뒤의 w와 손실은 각각 어떤 값에 가까워지나요?

답변: 

**Q4.** 이 결과에서 수렴은 무슨 뜻이며, 손실이 정확히 0이 되었나요?

답변:

---

## 기초 복습 과제

### 문제 3-1. 두 번의 업데이트 복습하기

### 문제 설명

교안 연습문제 1의 기본 계산을 복습합니다. 모델은 ŷ=wx+b, x=3, y=12입니다. 처음 w=2, b=1이고 학습률은 0.02입니다.

### 제공 공식

- 예측 ŷ=wx+b, 손실 L=(y−ŷ)²
- g_w=2(ŷ−y)x, g_b=2(ŷ−y)
- 새 w=w−0.02×g_w, 새 b=b−0.02×g_b

### 요구사항

- 초기 예측·손실·기울기를 계산합니다.
- 정확히 두 번 업데이트하고 매번 새 예측과 손실을 확인합니다.
- 문제 1-1의 순서와 코드를 참고해 숫자를 바꿔 계산하면 됩니다.
- 큰 학습률 비교나 추가 모델 구현은 요구하지 않습니다.

#### 해석 질문

**Q1.** 초기 예측, 손실, [w,b] 순서의 기울기는 얼마인가요?

**Q2.** 첫 업데이트 후 w, b, 예측, 손실은 얼마인가요?

**Q3.** 두 번째 기울기와 업데이트 후 w, b, 예측, 손실은 얼마인가요?

**Q4.** 왜 두 번째 업데이트에 첫 기울기를 그대로 사용하면 안 되나요?

#### 제출 결과

- 요구사항의 계산 결과 또는 제공 코드의 실행 결과
- Q1~Q4 답변

In [ ]:
# 문제 3-1. 두 번의 업데이트 복습하기
# 1. 문제의 초기값을 설정하세요.
# 2. 제공 공식에 숫자를 대입하세요.
# 3. 새 값과 결과를 출력하세요.

#### 답변 작성란

**Q1.** 초기 예측, 손실, [w,b] 순서의 기울기는 얼마인가요?

답변: 

**Q2.** 첫 업데이트 후 w, b, 예측, 손실은 얼마인가요?

답변: 

**Q3.** 두 번째 기울기와 업데이트 후 w, b, 예측, 손실은 얼마인가요?

답변: 

**Q4.** 왜 두 번째 업데이트에 첫 기울기를 그대로 사용하면 안 되나요?

답변:

### 문제 3-2. 검증 손실 표로 학습 종료 시점 찾기

### 문제 설명

교안 연습문제 3의 손실 표를 읽습니다. 학습 손실은 학습 데이터의 손실이고, 검증 손실은 따로 둔 검증 데이터의 손실입니다. 이 문제에서는 검증 손실이 작을수록 좋은 상태로 판단합니다.

| Epoch | 학습 손실 | 검증 손실 |
|---|---:|---:|
| 1 | 0.80 | 0.60 |
| 2 | 0.65 | 0.45 |
| 3 | 0.50 | 0.40 |
| 4 | 0.40 | 0.42 |
| 5 | 0.32 | 0.43 |
| 6 | 0.25 | 0.44 |

### 사용할 규칙

- 지금까지 가장 작은 검증 손실보다 **더 작아졌을 때** 개선으로 봅니다.
- 개선하면 ‘연속 미개선 횟수’를 0으로 되돌립니다.
- 개선하지 않으면 그 횟수를 1 더합니다.
- patience=3: 연속으로 3개 Epoch에서 개선되지 않으면 그 Epoch가 끝난 뒤 학습을 멈춥니다.
- Epoch 1은 첫 기준값으로 기록하며, 그때의 연속 미개선 횟수는 0입니다.

### 요구사항

- 검증 손실이 가장 작은 Epoch과 그 값을 찾습니다.
- Epoch 4, 5, 6의 연속 미개선 횟수를 셉니다.
- 학습을 멈추는 Epoch과 검증 손실이 가장 좋았던 Epoch을 구별합니다.
- 표를 읽어 답하면 됩니다. 실제 모델 학습이나 Early Stopping 프로그램 작성은 필수가 아닙니다.

#### 해석 질문

**Q1.** 검증 손실이 가장 작은 Epoch과 그 손실은 얼마인가요?

**Q2.** Epoch 4, 5, 6의 연속 미개선 횟수는 각각 얼마인가요?

**Q3.** patience=3이면 언제 학습을 멈추며, 가장 좋은 모델의 시점은 언제인가요?

**Q4.** 학습 손실이 계속 줄어도 이 규칙에서는 왜 학습을 멈추나요?

#### 제출 결과

- 요구사항의 계산 결과 또는 제공 코드의 실행 결과
- Q1~Q4 답변

In [ ]:
# 문제 3-2. 검증 손실 표로 학습 종료 시점 찾기
# 제공 공식에 숫자를 대입하고 결과를 확인하세요.
# 문제 3-2는 표를 읽어 아래 답변란에 작성하면 됩니다.

#### 답변 작성란

**Q1.** 검증 손실이 가장 작은 Epoch과 그 손실은 얼마인가요?

답변: 

**Q2.** Epoch 4, 5, 6의 연속 미개선 횟수는 각각 얼마인가요?

답변: 

**Q3.** patience=3이면 언제 학습을 멈추며, 가장 좋은 모델의 시점은 언제인가요?

답변: 

**Q4.** 학습 손실이 계속 줄어도 이 규칙에서는 왜 학습을 멈추나요?

답변:

---

## 실습 마무리

다음 질문으로 핵심 내용을 복습하세요.

**1.** 경사하강법의 기본 업데이트 식은 무엇인가요?

**2.** 학습률이 양수일 때 음수 기울기를 빼면 파라미터는 어떻게 되나요?

**3.** 학습률과 실제 변화량은 같은 값인가요?

**4.** Epoch와 Iteration은 어떻게 다른가요?

**5.** 수렴은 손실이 정확히 0이라는 뜻인가요?

**6.** Early Stopping의 종료 시점과 가장 좋은 검증 시점은 항상 같나요?

### 실습 마무리 답변 작성란

**1.** 답변: 

**2.** 답변: 

**3.** 답변: 

**4.** 답변: 

**5.** 답변: 

**6.** 답변: